In [1]:
# Run this cell first to install required packages
import subprocess, sys
packages = ['pymysql', 'requests', 'python-dotenv', 'groq', 'ipywidgets', 'rich']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages installed')

✅ All packages installed


In [2]:
import os
import sys
import json
import requests
from datetime import datetime
from collections import defaultdict

# ── Load .env from backend folder ────────────────────────────────────────
from dotenv import load_dotenv

# Adjust path if running from notebooks/ folder
env_path = os.path.join(os.path.dirname(os.getcwd()), 'backend', '.env')
if not os.path.exists(env_path):
    env_path = os.path.join(os.getcwd(), '..', 'backend', '.env')
load_dotenv(env_path)

# Add backend to path so we can import recommender, voting etc.
BACKEND_PATH = os.path.join(os.getcwd(), '..', 'backend')
if BACKEND_PATH not in sys.path:
    sys.path.insert(0, os.path.abspath(BACKEND_PATH))

# ── Config ────────────────────────────────────────────────────────────────
DB_CONFIG = {
    'host':     os.environ.get('MYSQLHOST',     'localhost'),
    'port':     int(os.environ.get('MYSQLPORT', 3306)),
    'user':     os.environ.get('MYSQLUSER',     'root'),
    'password': os.environ.get('MYSQLPASSWORD', ''),
    'database': os.environ.get('MYSQLDATABASE', 'railway'),
}

GROQ_API_KEY      = os.environ.get('GROQ_API_KEY', '')
OPENTRIPMAP_KEY   = os.environ.get('OPENTRIPMAP_KEY', '')
GROQ_URL          = 'https://api.groq.com/openai/v1/chat/completions'

print('✅ Config loaded')
print(f'   DB Host     : {DB_CONFIG["host"]}')
print(f'   Groq key    : {"✓ set" if GROQ_API_KEY else "✗ not set (will use rule-based fallback)"}')
print(f'   OpenTripMap : {"✓ set" if OPENTRIPMAP_KEY else "✗ not set (will use Wikipedia only)"}')

✅ Config loaded
   DB Host     : yamanote.proxy.rlwy.net
   Groq key    : ✗ not set (will use rule-based fallback)
   OpenTripMap : ✗ not set (will use Wikipedia only)


In [3]:
import pymysql

def get_db():
    return pymysql.connect(
        host        = DB_CONFIG['host'],
        port        = DB_CONFIG['port'],
        user        = DB_CONFIG['user'],
        password    = DB_CONFIG['password'],
        database    = DB_CONFIG['database'],
        cursorclass = pymysql.cursors.DictCursor
    )

def load_all_groups():
    """Load all groups with their vote counts."""
    conn = get_db()
    with conn.cursor() as cur:
        cur.execute("""
            SELECT g.group_id, g.name, g.created_at,
                   COUNT(v.id) as vote_count
            FROM groups_table g
            LEFT JOIN votes v ON g.group_id = v.group_id
            GROUP BY g.group_id
            ORDER BY g.created_at DESC
        """)
        groups = cur.fetchall()
    conn.close()
    return groups

def load_votes_for_group(group_id):
    """Load all votes for a specific group."""
    conn = get_db()
    with conn.cursor() as cur:
        cur.execute(
            'SELECT user_name, preferences FROM votes WHERE group_id = %s',
            (group_id,)
        )
        rows = cur.fetchall()
    conn.close()

    votes = {}
    for row in rows:
        prefs = row['preferences']
        if isinstance(prefs, str):
            try:
                prefs = json.loads(prefs)
            except:
                continue
        if isinstance(prefs, dict):
            votes[row['user_name']] = {'preferences': prefs}
    return votes

# Load and display all groups
groups = load_all_groups()
print(f'\n📊 Found {len(groups)} groups in database:\n')
print(f'{"#":<4} {"Group ID":<35} {"Name":<25} {"Votes":<8} {"Created"}')
print('-' * 90)
for i, g in enumerate(groups):
    print(f'{i+1:<4} {g["group_id"]:<35} {g["name"]:<25} {g["vote_count"]:<8} {str(g["created_at"])[:16]}')


📊 Found 28 groups in database:

#    Group ID                            Name                      Votes    Created
------------------------------------------------------------------------------------------
1    group_20260323152050                YJHD(Manali)              2        2026-03-23 09:50
2    group_20260323130428                Coorg                     2        2026-03-23 07:34
3    group_20260322204926                pANVHMARI 2.O             1        2026-03-22 15:19
4    group_20260322200325                GOA                       0        2026-03-22 14:33
5    group_20260322195912                lake.2                    0        2026-03-22 14:29
6    group_20260322195722                Panchmari 2.0             0        2026-03-22 14:27
7    group_20260322133359                pANVHMARI 2.O             0        2026-03-22 08:04
8    group_20260322133019                Buddies                   0        2026-03-22 08:00
9    group_20260322133016                Buddies

In [4]:
# ── SELECT WHICH GROUP TO ANALYZE ────────────────────────────────────────
# Change this to the group_id you want to generate itinerary for
# Leave as None to use the most recent group with votes

SELECTED_GROUP_ID = None   # e.g. 'group_20260319001639'

# Auto-select most recent group with votes if not specified
if not SELECTED_GROUP_ID:
    for g in groups:
        if g['vote_count'] > 0:
            SELECTED_GROUP_ID = g['group_id']
            print(f'Auto-selected: {SELECTED_GROUP_ID} ({g["name"]})')
            break

if not SELECTED_GROUP_ID:
    print('❌ No groups with votes found. Please create a group and submit votes first.')
else:
    votes = load_votes_for_group(SELECTED_GROUP_ID)
    print(f'\n✅ Loaded {len(votes)} votes for group: {SELECTED_GROUP_ID}')
    print(f'   Voters: {", ".join(votes.keys())}')

    # Show raw vote data
    print('\n📥 Raw vote data:')
    for voter, data in votes.items():
        prefs = data['preferences']
        print(f'  {voter}:')
        print(f'    Destinations : {prefs.get("destinations", [])}')
        print(f'    Budget       : {prefs.get("budget", "??")}')
        print(f'    Travel style : {prefs.get("travel_style", [])}')
        print(f'    Duration     : {prefs.get("duration", "?")} days')
        print(f'    Month        : {prefs.get("month", "??")}')    

Auto-selected: group_20260323152050 (YJHD(Manali))

✅ Loaded 2 votes for group: group_20260323152050
   Voters: Anvesha Gupta, Samriddhi Kumari

📥 Raw vote data:
  Anvesha Gupta:
    Destinations : ['manali']
    Budget       : medium
    Travel style : ['culture']
    Duration     : 5 days
    Month        : December
  Samriddhi Kumari:
    Destinations : ['Jaipur']
    Budget       : high
    Travel style : []
    Duration     : ? days
    Month        : ??


In [5]:
from collections import defaultdict

def calculate_consensus(votes: dict) -> dict:
    """Borda count algorithm — same as your voting.py."""
    destination_scores = defaultdict(float)
    budget_votes       = defaultdict(int)
    style_votes        = defaultdict(int)
    duration_votes     = []
    month_votes        = defaultdict(int)
    total_voters       = len(votes)

    for user, vote_data in votes.items():
        prefs = vote_data.get('preferences', {})
        destinations = prefs.get('destinations', [])
        n = len(destinations)
        for rank, dest in enumerate(destinations):
            points = n - rank
            destination_scores[dest] += points

        budget_votes[prefs.get('budget', 'medium')] += 1
        for style in prefs.get('travel_style', []):
            style_votes[style] += 1
        duration_votes.append(prefs.get('duration', 7))
        month_votes[prefs.get('month', 'December')] += 1

    if not destination_scores:
        return None

    sorted_dests = sorted(destination_scores.items(), key=lambda x: x[1], reverse=True)
    max_score    = max(destination_scores.values())
    top_destinations = [
        {'destination': d, 'score': s, 'percentage': round(s / max_score * 100)}
        for d, s in sorted_dests[:5]
    ]

    return {
        'winner':             top_destinations[0]['destination'],
        'top_destinations':   top_destinations,
        'consensus_budget':   max(budget_votes, key=budget_votes.get),
        'top_styles':         [s for s, _ in sorted(style_votes.items(), key=lambda x: x[1], reverse=True)[:3]],
        'avg_duration':       round(sum(duration_votes) / len(duration_votes)) if duration_votes else 7,
        'consensus_month':    max(month_votes, key=month_votes.get),
        'total_voters':       total_voters,
        'score_breakdown':    dict(destination_scores)
    }

# Run the voting engine
consensus = calculate_consensus(votes)

if consensus:
    print('\n' + '='*60)
    print('🏆  VOTING RESULTS')
    print('='*60)
    print(f'  Winner       : {consensus["winner"]}')
    print(f'  Budget       : {consensus["consensus_budget"]}')
    print(f'  Travel styles: {", ".join(consensus["top_styles"])}')
    print(f'  Duration     : {consensus["avg_duration"]} days')
    print(f'  Month        : {consensus["consensus_month"]}')
    print(f'  Total voters : {consensus["total_voters"]}')
    print('\n📊 Destination scores:')
    for d in consensus['top_destinations']:
        bar = '█' * (d['percentage'] // 5)
        print(f'  {d["destination"]:<20} {bar:<20} {d["percentage"]}%')
else:
    print('❌ No votes to calculate consensus.')


🏆  VOTING RESULTS
  Winner       : manali
  Budget       : medium
  Travel styles: culture
  Duration     : 6 days
  Month        : December
  Total voters : 2

📊 Destination scores:
  manali               ████████████████████ 100%
  Jaipur               ████████████████████ 100%


In [6]:
def fetch_wikipedia_summary(destination: str) -> str:
    """Get destination summary from Wikipedia (free, no key)."""
    try:
        url  = f'https://en.wikipedia.org/api/rest_v1/page/summary/{destination.replace(" ", "_")}'
        resp = requests.get(url, timeout=8)
        if resp.status_code == 200:
            return resp.json().get('extract', '')[:600]
    except Exception as e:
        print(f'  Wikipedia error: {e}')
    return f'{destination} is a popular travel destination in India.'

def fetch_attractions(destination: str) -> list:
    """Get tourist attractions via OpenTripMap (free key from opentripmap.com)."""
    if not OPENTRIPMAP_KEY:
        print('  ⚠ OpenTripMap key not set — skipping attractions fetch')
        return []
    try:
        geo_url = f'https://api.opentripmap.com/0.1/en/places/geoname?name={destination}&apikey={OPENTRIPMAP_KEY}'
        geo     = requests.get(geo_url, timeout=8).json()
        lat, lon = geo.get('lat'), geo.get('lon')
        if not lat:
            return []
        places_url = (
            f'https://api.opentripmap.com/0.1/en/places/radius'
            f'?radius=15000&lon={lon}&lat={lat}'
            f'&kinds=interesting_places,cultural,natural,religion,architecture'
            f'&limit=20&apikey={OPENTRIPMAP_KEY}'
        )
        places = requests.get(places_url, timeout=10).json()
        return [
            p['properties']['name']
            for p in places.get('features', [])
            if p['properties'].get('name')
        ][:12]
    except Exception as e:
        print(f'  OpenTripMap error: {e}')
        return []

def fetch_weather(destination: str) -> dict:
    """Get current weather from wttr.in (completely free)."""
    try:
        url  = f'https://wttr.in/{destination}?format=j1'
        resp = requests.get(url, timeout=8, headers={'User-Agent': 'PackVote/1.0'})
        if resp.status_code == 200:
            data    = resp.json()
            current = data['current_condition'][0]
            return {
                'temp_c':      current['temp_C'],
                'feels_like':  current['FeelsLikeC'],
                'humidity':    current['humidity'],
                'description': current['weatherDesc'][0]['value'],
                'forecast':    [
                    {
                        'date':    day['date'],
                        'max':     day['maxtempC'],
                        'min':     day['mintempC'],
                        'desc':    day['hourly'][4]['weatherDesc'][0]['value']
                    }
                    for day in data.get('weather', [])[:3]
                ]
            }
    except Exception as e:
        print(f'  Weather error: {e}')
    return {'temp_c': 'N/A', 'description': 'Weather data unavailable', 'forecast': []}

def fetch_local_events(destination: str) -> list:
    """Search Wikipedia for local festivals and events."""
    try:
        url    = f'https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={destination}+festival+fair+event&format=json&srlimit=5'
        resp   = requests.get(url, timeout=8).json()
        return [item['title'] for item in resp.get('query', {}).get('search', [])]
    except:
        return []

# ── Fetch everything ──────────────────────────────────────────────────────
DESTINATION = consensus['winner']
print(f'\n🔍 Fetching data for: {DESTINATION}\n')

print('  📖 Fetching Wikipedia summary...')
wiki_summary = fetch_wikipedia_summary(DESTINATION)
print(f'     {wiki_summary[:100]}...')

print('  🗺️  Fetching tourist attractions...')
attractions = fetch_attractions(DESTINATION)
print(f'     Found {len(attractions)} attractions: {", ".join(attractions[:5])}' if attractions else '     None found — AI will suggest from knowledge')

print('  🌤️  Fetching weather...')
weather = fetch_weather(DESTINATION)
print(f'     {weather["temp_c"]}°C — {weather["description"]}')

print('  🎪 Fetching local events...')
local_events = fetch_local_events(DESTINATION)
print(f'     {local_events}')

print('\n✅ Data fetch complete')


🔍 Fetching data for: manali

  📖 Fetching Wikipedia summary...
     manali is a popular travel destination in India....
  🗺️  Fetching tourist attractions...
  ⚠ OpenTripMap key not set — skipping attractions fetch
     None found — AI will suggest from knowledge
  🌤️  Fetching weather...
     -1°C — Moderate or heavy snow showers
  🎪 Fetching local events...
     []

✅ Data fetch complete


In [7]:
def build_prompt(destination, consensus, wiki_summary, attractions, local_events, weather) -> str:
    duration   = consensus['avg_duration']
    budget     = consensus['consensus_budget']
    styles     = ', '.join(consensus['top_styles'])
    month      = consensus['consensus_month']
    group_size = consensus['total_voters']
    attr_text  = ', '.join(attractions[:8]) if attractions else 'popular local spots'
    event_text = ', '.join(local_events[:3]) if local_events else 'local festivals'

    budget_daily = {'low': '₹1500', 'medium': '₹4000', 'high': '₹9000', 'luxury': '₹20000'}.get(budget, '₹4000')

    return f"""You are an expert Indian travel planner. Create a detailed {duration}-day itinerary for {destination}.

GROUP DETAILS:
- Group size   : {group_size} people
- Budget       : {budget} ({budget_daily}/person/day)
- Travel month : {month}
- Travel styles: {styles}
- Weather      : {weather.get('temp_c', 'N/A')}°C, {weather.get('description', '')}

DESTINATION CONTEXT:
- About        : {wiki_summary[:300]}
- Attractions  : {attr_text}
- Local events : {event_text}

Generate a PRACTICAL, DETAILED {duration}-day itinerary. Include:
1. Morning / Afternoon / Evening activities with timing
2. Specific restaurant recommendations (mention veg/nonveg, price range)
3. Local transport options (auto, bus, train, cab estimates)
4. Hotel/stay suggestions matching the budget
5. Daily cost estimate per person in INR
6. Pro tips locals would know
7. How to reach {destination} from major Indian cities (train/bus/flight)
8. Must-try local foods specific to {destination}
9. What to pack for {month} visit

Respond ONLY in this JSON format (no markdown, no extra text):
{{
  "destination": "{destination}",
  "duration": {duration},
  "theme": "catchy one-line trip description",
  "best_time": "best time to visit and why",
  "highlights": ["highlight 1", "highlight 2", "highlight 3"],
  "reach": {{
    "by_train": "nearest station + suggested trains from Delhi/Mumbai",
    "by_bus": "bus route suggestion",
    "by_flight": "nearest airport + airlines",
    "by_road": "road distance from nearest major city"
  }},
  "stay": {{
    "budget": "budget stay option with area name + price range",
    "mid": "mid-range option with area + price range",
    "luxury": "luxury option with name + price range"
  }},
  "days": [
    {{
      "day": 1,
      "title": "Day title",
      "morning": {{
        "time": "7:00 AM - 10:00 AM",
        "activity": "activity name",
        "description": "detailed 2-sentence description",
        "cost": "₹200 per person",
        "transport": "how to get there",
        "tips": "insider tip"
      }},
      "afternoon": {{
        "time": "11:00 AM - 3:00 PM",
        "activity": "activity name",
        "description": "detailed 2-sentence description",
        "cost": "₹300 per person",
        "transport": "how to get there",
        "tips": "insider tip"
      }},
      "evening": {{
        "time": "4:00 PM - 8:00 PM",
        "activity": "activity name",
        "description": "detailed 2-sentence description",
        "cost": "₹150 per person",
        "transport": "how to get there",
        "tips": "insider tip"
      }},
      "lunch": {{
        "restaurant": "restaurant name or type",
        "area": "locality name",
        "cuisine": "cuisine type",
        "type": "veg/nonveg/both",
        "must_try": "specific dish name",
        "price_for_two": "₹400"
      }},
      "dinner": {{
        "restaurant": "restaurant name or type",
        "area": "locality name",
        "cuisine": "cuisine type",
        "type": "veg/nonveg/both",
        "must_try": "specific dish name",
        "price_for_two": "₹500"
      }},
      "day_cost_per_person": "₹XXXX",
      "local_transport": "main transport for the day",
      "pro_tip": "one specific tip for today"
    }}
  ],
  "must_eat": [
    {{"dish": "dish name", "where": "where to find it", "price": "₹XX", "type": "veg/nonveg"}}
  ],
  "packing": ["packing item 1", "packing item 2", "packing item 3", "packing item 4"],
  "safety_tips": ["tip 1", "tip 2"],
  "useful_apps": ["app 1 - why useful", "app 2 - why useful"],
  "total_cost": {{
    "per_person_inr":  "₹XXXXX",
    "for_group_inr":   "₹XXXXX",
    "breakdown": {{
      "accommodation": "₹XXXX",
      "food":          "₹XXXX",
      "transport":     "₹XXXX",
      "activities":    "₹XXXX",
      "miscellaneous": "₹XXXX"
    }}
  }}
}}"""


def call_groq(prompt: str) -> dict:
    """Call Groq API with LLaMA 3 70B (free tier)."""
    headers = {
        'Authorization': f'Bearer {GROQ_API_KEY}',
        'Content-Type':  'application/json'
    }
    payload = {
        'model':       'llama3-70b-8192',
        'max_tokens':  4096,
        'temperature': 0.7,
        'messages': [
            {
                'role':    'system',
                'content': 'You are an expert Indian travel planner with deep knowledge of every state, city, town and village in India. Always respond with valid JSON only.'
            },
            {'role': 'user', 'content': prompt}
        ]
    }
    resp = requests.post(GROQ_URL, headers=headers, json=payload, timeout=60)
    raw  = resp.json()['choices'][0]['message']['content'].strip()

    # Strip markdown code fences if present
    if raw.startswith('```'):
        lines = raw.split('\n')
        raw   = '\n'.join(lines[1:-1] if lines[-1] == '```' else lines[1:])
    return json.loads(raw)


def rule_based_itinerary(destination, consensus, attractions, weather) -> dict:
    """Fallback itinerary when Groq is not available."""
    duration = consensus['avg_duration']
    budget   = consensus['consensus_budget']
    daily_costs = {'low': 1500, 'medium': 4000, 'high': 9000, 'luxury': 20000}
    daily_cost  = daily_costs.get(budget, 4000)

    # Use attractions if available, else generic
    spots = attractions[:12] if attractions else [
        'City Museum', 'Central Market', 'Historical Fort',
        'Local Temple', 'Garden', 'Viewpoint', 'Art Gallery'
    ]

    days = []
    slot_idx = 0
    day_themes = [
        'Arrival & First Impressions',
        'Heritage & Culture',
        'Nature & Adventure',
        'Food & Local Life',
        'Shopping & Leisure',
        'Hidden Gems',
        'Departure Day'
    ]

    for d in range(1, duration + 1):
        morning_spot   = spots[slot_idx % len(spots)];   slot_idx += 1
        afternoon_spot = spots[slot_idx % len(spots)];   slot_idx += 1
        evening_spot   = spots[slot_idx % len(spots)];   slot_idx += 1

        days.append({
            'day':   d,
            'title': day_themes[(d - 1) % len(day_themes)],
            'morning':   {'time': '8:00 AM', 'activity': morning_spot,   'description': f'Explore {morning_spot}',   'cost': f'₹{int(daily_cost * 0.2)}', 'transport': 'Auto/cab', 'tips': 'Go early to avoid crowds'},
            'afternoon': {'time': '12:00 PM', 'activity': afternoon_spot, 'description': f'Visit {afternoon_spot}',  'cost': f'₹{int(daily_cost * 0.3)}', 'transport': 'Auto/walk', 'tips': 'Carry water'},
            'evening':   {'time': '5:00 PM',  'activity': evening_spot,   'description': f'Evening at {evening_spot}','cost': f'₹{int(daily_cost * 0.15)}', 'transport': 'Walk', 'tips': 'Great for photos'},
            'lunch':     {'restaurant': 'Local dhaba', 'area': 'City centre', 'cuisine': 'Regional', 'type': 'both', 'must_try': 'Local thali', 'price_for_two': f'₹{int(daily_cost * 0.15)}'},
            'dinner':    {'restaurant': 'Local restaurant', 'area': 'Main market', 'cuisine': 'Regional', 'type': 'both', 'must_try': 'Street food speciality', 'price_for_two': f'₹{int(daily_cost * 0.2)}'},
            'day_cost_per_person': f'₹{daily_cost}',
            'local_transport':     'Auto rickshaw / local bus / walking',
            'pro_tip':             f'Day {d}: Ask locals for the best hidden spots'
        })

    return {
        'destination': destination,
        'duration':    duration,
        'theme':       f'Discover the best of {destination}',
        'highlights':  [f'Explore {s}' for s in spots[:3]],
        'reach': {
            'by_train':  f'Search trains to nearest station to {destination} on IRCTC',
            'by_bus':    f'RedBus routes available to {destination}',
            'by_flight': f'Search flights to nearest airport via Google Flights',
            'by_road':   'Check Google Maps for road route'
        },
        'days':       days,
        'must_eat':   [{'dish': 'Local speciality', 'where': 'City centre', 'price': '₹100', 'type': 'both'}],
        'packing':    [f'Light clothes for {consensus["consensus_month"]}', 'Comfortable shoes', 'Sunscreen', 'Power bank'],
        'total_cost': {
            'per_person_inr': f'₹{daily_cost * duration + 8000}',
            'for_group_inr':  f'₹{(daily_cost * duration + 8000) * consensus["total_voters"]}',
            'breakdown': {
                'accommodation': f'₹{int(daily_cost * 0.35 * duration)}',
                'food':          f'₹{int(daily_cost * 0.30 * duration)}',
                'transport':     f'₹{int(daily_cost * 0.15 * duration + 4000)}',
                'activities':    f'₹{int(daily_cost * 0.20 * duration)}',
                'miscellaneous': f'₹{int(daily_cost * 0.10 * duration)}'
            }
        },
        'ai_powered': False
    }


# ── Generate the itinerary ────────────────────────────────────────────────
print(f'\n🤖 Generating itinerary for {DESTINATION}...')
print(f'   Duration : {consensus["avg_duration"]} days')
print(f'   Budget   : {consensus["consensus_budget"]}')
print(f'   Month    : {consensus["consensus_month"]}')
print(f'   Styles   : {", ".join(consensus["top_styles"])}')

if GROQ_API_KEY:
    print('\n  Calling Groq API (LLaMA 3 70B)... this takes ~15 seconds')
    try:
        prompt    = build_prompt(DESTINATION, consensus, wiki_summary, attractions, local_events, weather)
        itinerary = call_groq(prompt)
        itinerary['ai_powered'] = True
        itinerary['ai_model']   = 'LLaMA 3 70B via Groq'
        print('  ✅ AI itinerary generated!')
    except Exception as e:
        print(f'  ⚠ Groq failed: {e}')
        print('  Falling back to rule-based itinerary...')
        itinerary = rule_based_itinerary(DESTINATION, consensus, attractions, weather)
else:
    print('  ⚠ No Groq key — using rule-based itinerary')
    itinerary = rule_based_itinerary(DESTINATION, consensus, attractions, weather)

print(f'\n✅ Itinerary ready — {len(itinerary.get("days", []))} days generated')
print(f'   AI powered: {itinerary.get("ai_powered", False)}')


🤖 Generating itinerary for manali...
   Duration : 6 days
   Budget   : medium
   Month    : December
   Styles   : culture
  ⚠ No Groq key — using rule-based itinerary

✅ Itinerary ready — 6 days generated
   AI powered: False


In [8]:
from IPython.display import display, HTML

def render_itinerary_html(itinerary, consensus, weather):
    dest     = itinerary.get('destination', '')
    ai_badge = '<span style="background:#6c63ff;color:white;padding:3px 10px;border-radius:12px;font-size:11px;margin-left:8px;">✨ AI Generated</span>' if itinerary.get('ai_powered') else ''

    html = f"""
    <style>
      .itin {{ font-family: Arial, sans-serif; max-width: 900px; color: #222; }}
      .itin h1 {{ font-size: 26px; color: #2c2c2c; border-bottom: 3px solid #6c63ff; padding-bottom: 10px; }}
      .itin h2 {{ font-size: 18px; color: #6c63ff; margin-top: 28px; }}
      .itin h3 {{ font-size: 15px; color: #333; margin: 6px 0; }}
      .day-card {{ border: 1px solid #e0e0e0; border-radius: 12px; margin: 16px 0; overflow: hidden; }}
      .day-header {{ background: linear-gradient(135deg, #6c63ff, #ff6584); color: white; padding: 12px 18px; }}
      .day-body {{ padding: 16px 18px; }}
      .slot {{ border-left: 4px solid #6c63ff; padding: 10px 14px; margin: 10px 0; background: #f9f9ff; border-radius: 0 8px 8px 0; }}
      .slot-morning {{ border-color: #ff9800; background: #fff8f0; }}
      .slot-afternoon {{ border-color: #4caf50; background: #f0fff4; }}
      .slot-evening {{ border-color: #9c27b0; background: #fdf4ff; }}
      .meal {{ display: inline-block; background: #fff; border: 1px solid #e0e0e0; border-radius: 8px; padding: 8px 12px; margin: 6px 4px; font-size: 13px; }}
      .veg {{ border-left: 4px solid #4caf50; }}
      .nonveg {{ border-left: 4px solid #f44336; }}
      .both {{ border-left: 4px solid #ff9800; }}
      .chip {{ display: inline-block; background: #f0f0ff; color: #6c63ff; border-radius: 20px; padding: 4px 12px; margin: 3px; font-size: 12px; }}
      .cost-box {{ background: linear-gradient(135deg, #ffd200, #ff6584); color: white; border-radius: 10px; padding: 14px 18px; margin: 8px 0; }}
      .stat {{ display: inline-block; background: #f5f5ff; border-radius: 8px; padding: 8px 14px; margin: 4px; font-size: 13px; }}
      .reach-card {{ background: #f0f4ff; border-radius: 10px; padding: 14px 18px; margin: 8px 0; }}
      .food-card {{ background: #fff8f0; border: 1px solid #ffe0b2; border-radius: 10px; padding: 12px 16px; margin: 8px 4px; display: inline-block; min-width: 200px; vertical-align: top; }}
      .weather-badge {{ background: #e3f2fd; color: #1565c0; border-radius: 8px; padding: 6px 12px; font-size: 13px; display: inline-block; }}
    </style>
    <div class='itin'>
    <h1>🗺️ {dest} — {itinerary.get('duration', '')} Day Itinerary {ai_badge}</h1>
    <p style='font-style:italic;color:#666;font-size:15px;'>"{itinerary.get('theme', '')}"</p>
    """

    # Weather & stats
    html += f"""
    <div style='margin:12px 0;'>
      <span class='stat'>👥 {consensus['total_voters']} voters</span>
      <span class='stat'>💰 {consensus['consensus_budget'].capitalize()} budget</span>
      <span class='stat'>📅 {consensus['consensus_month']}</span>
      <span class='stat'>🧳 {', '.join(consensus['top_styles'])}</span>
      <span class='weather-badge'>🌤 {weather.get('temp_c', 'N/A')}°C · {weather.get('description', '')}</span>
    </div>
    """

    # Highlights
    if itinerary.get('highlights'):
        html += '<h2>⭐ Trip Highlights</h2>'
        for h in itinerary['highlights']:
            html += f'<span class="chip">★ {h}</span>'

    # How to reach
    reach = itinerary.get('reach', {})
    if reach:
        html += '<h2>🚂 How to Reach</h2><div class="reach-card">'
        icons = {'by_train': '🚆', 'by_bus': '🚌', 'by_flight': '✈️', 'by_road': '🚗'}
        labels = {'by_train': 'Train', 'by_bus': 'Bus', 'by_flight': 'Flight', 'by_road': 'Road'}
        for key, icon in icons.items():
            if reach.get(key):
                html += f'<p style="margin:6px 0;"><strong>{icon} {labels[key]}:</strong> {reach[key]}</p>'
        html += '</div>'

    # Stay options
    stay = itinerary.get('stay', {})
    if stay:
        html += '<h2>🏨 Where to Stay</h2>'
        for tier, desc in stay.items():
            color = {'budget': '#4caf50', 'mid': '#ff9800', 'luxury': '#9c27b0'}.get(tier, '#6c63ff')
            html += f'<div style="border-left:4px solid {color};padding:6px 12px;margin:6px 0;background:#fafafa;border-radius:0 8px 8px 0;"><strong>{tier.capitalize()}:</strong> {desc}</div>'

    # Day-by-day
    html += '<h2>📅 Day-by-Day Itinerary</h2>'
    for day in itinerary.get('days', []):
        html += f"""
        <div class='day-card'>
          <div class='day-header'>
            <strong>Day {day['day']}: {day.get('title', '')}</strong>
            &nbsp;·&nbsp; {day.get('day_cost_per_person', '')} per person
            &nbsp;·&nbsp; 🚗 {day.get('local_transport', '')}
          </div>
          <div class='day-body'>
        """

        # Morning / Afternoon / Evening
        for slot_key, slot_class, slot_icon in [
            ('morning',   'slot-morning',   '🌅'),
            ('afternoon', 'slot-afternoon', '☀️'),
            ('evening',   'slot-evening',   '🌇')
        ]:
            slot = day.get(slot_key, {})
            if slot:
                html += f"""
                <div class='slot {slot_class}'>
                  <div style='font-size:12px;color:#888;'>{slot_icon} {slot.get('time', slot_key.capitalize())}</div>
                  <h3>{slot.get('activity', '')}</h3>
                  <p style='margin:4px 0;font-size:13px;color:#555;'>{slot.get('description', '')}</p>
                  <div style='font-size:12px;margin-top:6px;'>
                    💰 {slot.get('cost', '')} &nbsp;·&nbsp;
                    🚗 {slot.get('transport', '')} &nbsp;·&nbsp;
                    💡 {slot.get('tips', '')}
                  </div>
                </div>"""

        # Meals
        html += '<div style="margin-top:10px;">'
        for meal_key, meal_icon in [('lunch', '🍽️ Lunch'), ('dinner', '🌙 Dinner')]:
            meal = day.get(meal_key, {})
            if meal:
                mtype = meal.get('type', 'both')
                html += f"""
                <div class='meal {mtype}'>
                  <div style='font-size:11px;color:#888;'>{meal_icon}</div>
                  <strong>{meal.get('restaurant', '')}</strong><br>
                  <span style='font-size:12px;color:#555;'>{meal.get('area', '')} · {meal.get('cuisine', '')}</span><br>
                  <span style='font-size:12px;'>Try: <em>{meal.get('must_try', '')}</em> · {meal.get('price_for_two', '')}</span>
                </div>"""
        html += '</div>'

        if day.get('pro_tip'):
            html += f'<div style="background:#fffbe6;border-radius:8px;padding:8px 12px;margin-top:8px;font-size:13px;">💡 <strong>Pro tip:</strong> {day["pro_tip"]}</div>'

        html += '</div></div>'

    # Must eat
    must_eat = itinerary.get('must_eat', [])
    if must_eat:
        html += '<h2>🍜 Must-Try Local Food</h2>'
        for food in must_eat:
            if isinstance(food, dict):
                color = '#4caf50' if food.get('type') == 'veg' else '#f44336' if food.get('type') == 'nonveg' else '#ff9800'
                html += f"""
                <div class='food-card'>
                  <strong style='color:{color};'>{food.get('dish', food)}</strong><br>
                  <span style='font-size:12px;color:#666;'>📍 {food.get('where', '')} · {food.get('price', '')}</span>
                </div>"""
            else:
                html += f'<span class="chip">🍽 {food}</span>'

    # Cost breakdown
    cost = itinerary.get('total_cost', {})
    if cost:
        html += f"""
        <h2>💰 Total Cost Estimate</h2>
        <div class='cost-box'>
          <div style='font-size:22px;font-weight:bold;'>{cost.get('per_person_inr', '')} per person</div>
          <div style='font-size:14px;opacity:0.9;margin-top:4px;'>{cost.get('for_group_inr', '')} total for group of {consensus['total_voters']}</div>
        </div>"""
        breakdown = cost.get('breakdown', {})
        if breakdown:
            html += '<div style="margin-top:10px;">'
            icons_map = {'accommodation': '🏨', 'food': '🍜', 'transport': '🚗', 'activities': '🎯', 'miscellaneous': '💼'}
            for k, v in breakdown.items():
                html += f'<span class="stat">{icons_map.get(k, "")} {k.capitalize()}: <strong>{v}</strong></span>'
            html += '</div>'

    # Packing
    packing = itinerary.get('packing', [])
    if packing:
        html += '<h2>🎒 What to Pack</h2>'
        for p in packing:
            html += f'<span class="chip">✓ {p}</span>'

    # Safety tips
    safety = itinerary.get('safety_tips', [])
    if safety:
        html += '<h2>⚠️ Safety Tips</h2><ul style="color:#555;font-size:14px;">'
        for s in safety:
            html += f'<li>{s}</li>'
        html += '</ul>'

    # Useful apps
    apps = itinerary.get('useful_apps', [])
    if apps:
        html += '<h2>📱 Useful Apps</h2>'
        for a in apps:
            html += f'<span class="chip">📲 {a}</span>'

    html += '</div>'
    return html

# Render
display(HTML(render_itinerary_html(itinerary, consensus, weather)))

In [9]:
def save_itinerary_to_db(group_id, itinerary, consensus):
    """Save the generated itinerary to the itineraries table."""
    try:
        conn = get_db()
        with conn.cursor() as cur:
            # Create table if not exists
            cur.execute("""
                CREATE TABLE IF NOT EXISTS itineraries (
                    id           INT AUTO_INCREMENT PRIMARY KEY,
                    group_id     VARCHAR(50) NOT NULL,
                    destination  VARCHAR(150),
                    duration     INT,
                    itinerary    JSON,
                    ai_powered   BOOLEAN DEFAULT FALSE,
                    created_at   DATETIME DEFAULT CURRENT_TIMESTAMP
                )
            """)
            cur.execute(
                """INSERT INTO itineraries (group_id, destination, duration, itinerary, ai_powered)
                   VALUES (%s, %s, %s, %s, %s)""",
                (
                    group_id,
                    itinerary.get('destination', ''),
                    itinerary.get('duration', 0),
                    json.dumps(itinerary),
                    itinerary.get('ai_powered', False)
                )
            )
        conn.commit()
        conn.close()
        print(f'✅ Itinerary saved to database for group {group_id}')
        return True
    except Exception as e:
        print(f'❌ Failed to save: {e}')
        return False

# Save to DB
save_itinerary_to_db(SELECTED_GROUP_ID, itinerary, consensus)

❌ Failed to save: (1054, "Unknown column 'itinerary' in 'field list'")


False

In [10]:
# Save itinerary as JSON file
filename = f'itinerary_{DESTINATION.replace(" ", "_")}_{datetime.now().strftime("%Y%m%d_%H%M")}.json'

output = {
    'generated_at':  datetime.now().isoformat(),
    'group_id':      SELECTED_GROUP_ID,
    'consensus':     consensus,
    'weather':       weather,
    'itinerary':     itinerary
}

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'✅ Itinerary saved as: {filename}')
print(f'   File size: {os.path.getsize(filename) / 1024:.1f} KB')
print(f'\n📋 Summary:')
print(f'   Destination : {itinerary["destination"]}')
print(f'   Duration    : {itinerary["duration"]} days')
print(f'   AI powered  : {itinerary.get("ai_powered", False)}')
print(f'   Total cost  : {itinerary.get("total_cost", {}).get("per_person_inr", "N/A")} per person')

✅ Itinerary saved as: itinerary_manali_20260323_1640.json
   File size: 11.2 KB

📋 Summary:
   Destination : manali
   Duration    : 6 days
   AI powered  : False
   Total cost  : ₹32000 per person
